In [1]:
import importlib
import Base
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler 
import warnings
from collections import Counter
import collections
import pickle 
import numpy as np
from sklearn.decomposition import PCA


In [2]:
warnings.filterwarnings('ignore')

importlib.reload(Base)
d2v = Base.getd2v()
d2vDf = Base.getd2vDf()
indGrpNames = Base.getindGrpNames()
longD = Base.getlongD()


In [3]:
warnings.filterwarnings('ignore')

kmDf = d2vDf.copy() # Create a new dataframe kmDf as a copy of d2vDf

Y = d2v.docvecs.index2entity # From the doc2vec model get the tags, which are the clean descriptions 

X = [] 
for tag in Y:
    X.append(d2v.docvecs[tag]) # Document vectors from the doc2vec model to matrix

scaler = StandardScaler() # Standardize all the loadings to ~N(0,1)
scaled_X = scaler.fit_transform(X)

km_model = KMeans(n_clusters=24, max_iter=10000, tol=1e-8) # Train the KMeans model
km_model.fit(scaled_X)
pred = km_model.labels_  # Get cluster assignment labels


d2vIndGrp = pd.DataFrame( # Merge the industry groups to the dataframe
    {'cleanDesc': Y,
     'd2vIndGrp': pred
    })

kmDf = kmDf.merge(d2vIndGrp, on='cleanDesc', how='inner')

print(kmDf.groupby('d2vIndGrp').size()) # Count the number of companies in each cluster (d2vIndGrp).


d2vIndGrp
0      44
1     121
2      34
3      93
4     125
5      29
6      72
7      36
8       9
9      35
10     16
11     30
12     16
13     80
14      8
15     22
16     46
17      5
18     53
19     31
20     16
21      3
22     63
23     13
dtype: int64


In [4]:
def wordFrequency(iTxt, mcap):
    d=dict(Counter(iTxt.split()))
    k={i:(j/100)*mcap for i,j in d.items()}
    return k

topicnames = []
for i in range(0,24):
    print("")
    print("")
    print("####### KMeans Cluster " + str(i) + " ########")
    smallD2 = kmDf[ kmDf.d2vIndGrp==i ]
    smallD2['wF'] = smallD2.apply(lambda x:  wordFrequency(x.cleanDesc, x.logmcap), axis=1)

    print(smallD2[['compustat_name','mktcap']].head(n=10))
    names = []
    for ii, n in smallD2.head(n=5).iterrows():
        names.append(n.compustat_name)

    wF = smallD2['wF'].to_list() 
    counter = collections.Counter()
    for d in wF: 
        counter.update(d)
    result = pd.DataFrame(dict(counter).items(), columns=['word', 'importance'])
    result = result.sort_values(by='importance',ascending=False)
    words = []
    for ii, n in result.head(n=5).iterrows():
        words.append(n.word)
    topicnames.append([i, names, words])
    print(words)

topicnames = pd.DataFrame(topicnames, columns=['d2vIndGrp','top5companies','top5words'])
print(topicnames)




####### KMeans Cluster 0 ########
                   compustat_name      mktcap
12        APELLIS PHARMACEUTICALS  4541.00760
41            KODIAK SCIENCES INC  4384.56726
63   INTRA-CELLULAR THERAPIES INC  4263.77342
138        KARUNA THERPEUTICS INC  3880.74400
153  ZENTALIS PHARMACEUTICALS INC  3808.42236
169    ACADIA PHARMACEUTICALS INC  3752.04504
252               OPKO HEALTH INC  3277.25021
262                    INSMED INC  3224.45328
264       AMICUS THERAPEUTICS INC  3218.28045
356         ARCUS BIOSCIENCES INC  2842.97703
['treatment', 'patient', 'patent', 'study', 'phase']


####### KMeans Cluster 1 ########
                compustat_name      mktcap
9          MERITAGE HOMES CORP  4554.18066
24  HANNON ARMSTRONG SUST INFR  4498.94528
28       PARK HOTELS & RESORTS  4464.72352
29          COHEN & STEERS INC  4464.25507
32     RYAN SPECIALITY GRP INC  4434.62640
56  FIRST NATL OF NEBRASKA INC  4296.43203
57   TAYLOR MORRISON HOME CORP  4296.26936
59          COLONY CAPITA

['pipeline', 'crude_oil', 'refinery', 'terminal', 'natural_gas']


####### KMeans Cluster 9 ########
                   compustat_name      mktcap
49                    ARVINAS INC  4342.90608
54      IONIS PHARMACEUTICALS INC  4297.02030
150              CYTOKINETICS INC  3823.38714
242        RELAY THERAPEUTICS INC  3318.03124
256        KYMERA THERAPEUTIC INC  3262.56063
297  SPRINGWORKS THERAPEUTICS INC  3052.45302
316   IOVANCE BIOTHERAPEUTICS INC  2995.20191
339         SANA BIOTCHNOLOGY INC  2923.13484
388           TG THERAPEUTICS INC  2715.91700
417     LIGAND PHARMACEUTICAL INC  2581.33552
['patient', 'patent', 'treatment', 'cell', 'phase']


####### KMeans Cluster 10 ########
                   compustat_name      mktcap
30                      CHEGG INC  4449.87290
58                   DUOLINGO INC  4286.31345
142                 BLACKBAUD INC  3860.77934
173                    NELNET INC  3735.38088
213    GRAND CANYON EDUCATION INC  3434.74254
230      INSTRUCTURE HOLDING

['software', 'device', 'vehicle', 'cloud', 'power']


####### KMeans Cluster 14 ########
                  compustat_name      mktcap
221         RED ROCK RESORTS INC  3400.22311
295                    ST JOE CO  3064.86015
355              CEDAR FAIR  -LP  2845.51052
461      MADISON SQUARE GARD ENT  2405.06528
546                  BALLYS CORP  2069.05578
592           EVERI HOLDINGS INC  1942.40165
809     GOLDEN ENTERTAINMENT INC  1468.25021
846  MONARCH CASINO & RESORT INC  1385.52720
['gaming', 'casino', 'park', 'game', 'entertainment']


####### KMeans Cluster 15 ########
                compustat_name      mktcap
6            INARI MEDICAL INC  4582.21035
46           STAAR SURGICAL CO  4345.88000
84                 CONMED CORP  4146.05472
201  MERIT MEDICAL SYSTEMS INC  3517.33340
268               ATRICURE INC  3193.65196
360                 NEVRO CORP  2829.66728
363      INTEGER HOLDINGS CORP  2826.09621
390               NUVASIVE INC  2715.10528
393           HAEMONETICS CO

In [5]:
for i, n in indGrpNames.iterrows():
    print("")
    print("")
    print("####### GICS Industry Group " + str(n.gicsIndGrp) + ": "+n.indGrp+" ########")
    print()
    ntopics = kmDf[kmDf.gicsIndGrp==n.gicsIndGrp].groupby(['d2vIndGrp']).size().reset_index()
    ntopics.columns = ['d2vIndGrp','n']
    ntopics['pct'] = round(100.0*ntopics['n'] / ntopics['n'].sum(),1 )
    ntopics = ntopics.merge(topicnames, on='d2vIndGrp')
    ntopics = ntopics.sort_values(by='n',ascending=False)
    print(ntopics[['d2vIndGrp','n','pct','top5words']])




####### GICS Industry Group 1010: Energy ########

   d2vIndGrp   n   pct                                          top5words
1          3  30  54.5  [energy, project, aircraft, transportation, na...
3         11  10  18.2  [natural_gas, pipeline, energy, storage, capac...
2          8   9  16.4  [pipeline, crude_oil, refinery, terminal, natu...
4         16   4   7.3    [power, component, vehicle, equipment, battery]
0          1   2   3.6          [insurance, home, loan, investment, care]


####### GICS Industry Group 1510: Materials ########

   d2vIndGrp   n   pct                                          top5words
2          6  24  54.5  [equipment, industrial, brand, construction, b...
4         22   8  18.2               [brand, food, store, retailer, home]
1          3   6  13.6  [energy, project, aircraft, transportation, na...
0          1   3   6.8          [insurance, home, loan, investment, care]
3         11   3   6.8  [natural_gas, pipeline, energy, storage, capac...


#

In [6]:
for c in range(0,24):
    print("")
    print("")
    print("####### KMeans Cluster " + str(c) + ": "+str(topicnames.top5words.iloc[c])+" ########")
    print()
    ntopics = kmDf[kmDf.d2vIndGrp==c].groupby(['gicsIndGrp']).size().reset_index()
    ntopics.columns = ['gicsIndGrp','n']
    ntopics['pct'] = round(100.0*ntopics['n'] / ntopics['n'].sum(),1 )
    ntopics = ntopics.merge(indGrpNames, on='gicsIndGrp')
    ntopics = ntopics.sort_values(by='n',ascending=False)
    print(ntopics[['gicsIndGrp','n','pct','indGrp']])
    



####### KMeans Cluster 0: ['treatment', 'patient', 'patent', 'study', 'phase'] ########

   gicsIndGrp   n   pct                                          indGrp
1        3520  43  97.7  Pharmaceuticals, Biotechnology & Life Sciences
0        3510   1   2.3                Health Care Equipment & Services


####### KMeans Cluster 1: ['insurance', 'home', 'loan', 'investment', 'care'] ########

    gicsIndGrp   n   pct                                          indGrp
9         3510  19  15.7                Health Care Equipment & Services
19        6010  13  10.7                                     Real Estate
13        4030  10   8.3                                       Insurance
11        4010  10   8.3                                           Banks
3         2020   9   7.4              Commercial & Professional Services
12        4020   9   7.4                          Diversified Financials
14        4510   8   6.6                             Software & Services
5         2520   6 

    gicsIndGrp   n   pct                                          indGrp
8         3020  15  23.8                        Food, Beverage & Tobacco
9         3030   9  14.3                   Household & Personal Products
0         1510   8  12.7                                       Materials
6         2550   7  11.1                                       Retailing
7         3010   6   9.5                        Food & Staples Retailing
5         2530   5   7.9                               Consumer Services
2         2020   3   4.8              Commercial & Professional Services
4         2520   3   4.8                     Consumer Durables & Apparel
1         2010   2   3.2                                   Capital Goods
3         2510   2   3.2                        Automobiles & Components
11        3520   2   3.2  Pharmaceuticals, Biotechnology & Life Sciences
10        3510   1   1.6                Health Care Equipment & Services


####### KMeans Cluster 23: ['brand', 'food', 'pro

In [7]:
def wordFrequency(col):
    sdFreq = kmDf[col].str.split(expand=True).stack().value_counts()
    return sdFreq

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    print(wordFrequency('cleanDesc')[0:100])
    

loan                  3838
patient               2749
brand                 2434
bank                  2095
software              2080
patent                2078
security              1942
client                1868
equipment             1782
home                  1719
treatment             1709
investment            1600
payment               1564
provider              1559
care                  1500
insurance             1492
financial             1447
banking               1431
industrial            1401
vehicle               1399
device                1363
clinical              1291
store                 1283
energy                1261
power                 1244
online                1234
cell                  1212
pipeline              1209
construction          1169
study                 1136
component             1118
mortgage              1095
content               1064
project               1058
phase                 1052
user                  1030
enterprise            1001
m

In [8]:
warnings.filterwarnings('ignore')

kmDf2 = d2vDf.copy() # First make a copy to a new dataframe

Y = []
for i, k in kmDf2.iterrows():
    Y.append(k.companyid)

vec = []
for f in longD['cleanDesc']:
    vec.append(d2v.docvecs[f])
    
pca24 = PCA(n_components=24)
pca24_result = pca24.fit_transform(vec)
print(pca24_result)

scaler = StandardScaler() # Standardize all the loadings to ~N(0,1)
scaled_X = scaler.fit_transform(np.array(pca24_result))   

km_model2 = KMeans(n_clusters=24, max_iter=10000, tol=1e-8)
km_model2.fit(scaled_X)
pred2 = km_model2.labels_   # Get cluster assignment labels

d2vIndGrpPca = pd.DataFrame( # Merge the industry groups to the dataframe
    {'companyid': Y,
     'd2vIndGrpPca': pred2
    })

kmDf2 = kmDf2.merge(d2vIndGrpPca, on='companyid')

print(kmDf2.groupby('d2vIndGrpPca').size()) # Count the number of companies in each cluster (d2vIndGrpPca).


[[-0.11671631 -2.04984856 -1.52498964 ...  0.17086921  0.01632785
   0.10280752]
 [-3.48533081  4.61931617 -1.32386966 ...  0.54468546  0.01517507
   0.21945509]
 [-0.53141571 -2.90724801 -2.17745848 ...  0.14693727  0.21118789
   0.33502724]
 ...
 [-2.5313842  -3.66184629  1.71987321 ...  0.30057519 -0.38309678
   0.15356868]
 [-0.02266791  2.46286376  0.86520023 ... -0.11329618 -0.09482739
   0.02412679]
 [-0.98110542 -4.71481392 -1.99711827 ...  0.01999411 -0.5386714
  -0.96638163]]
d2vIndGrpPca
0     325
1      20
2     137
3      18
4       1
5      19
6       1
7      32
8       8
9      14
10     37
11    110
12     37
13     16
14    109
15      1
16      1
17      1
18     15
19     44
20     24
21     27
22      4
23      1
dtype: int64


In [9]:
def wordFrequency(iTxt, mcap):
    d=dict(Counter(iTxt.split()))
    k={i:(j/100)*mcap for i,j in d.items()}
    return k

topicnames = []
for i in range(0,24):
    print("")
    print("")
    print("####### KMeans Cluster " + str(i) + " ########")
    smallD2 = kmDf2[ kmDf2.d2vIndGrpPca == i ]
    smallD2['wF'] = smallD2.apply(lambda x:  wordFrequency(x.cleanDesc, x.logmcap), axis=1)

    print(smallD2[['compustat_name','mktcap']].head(n=10))
    names = []
    for ii, n in smallD2.head(n=5).iterrows():
        names.append(n.compustat_name)

    wF = smallD2['wF'].to_list() 
    counter = collections.Counter()
    for d in wF: 
        counter.update(d)
    result = pd.DataFrame(dict(counter).items(), columns=['word', 'importance'])
    result = result.sort_values(by='importance',ascending=False)
    words = []
    for ii, n in result.head(n=5).iterrows():
        words.append(n.word)
    topicnames.append([i, names, words])
    print(words)

topicnames = pd.DataFrame(topicnames, columns=['d2vIndGrpPca','top5companies','top5words'])
print(topicnames)




####### KMeans Cluster 0 ########
                compustat_name      mktcap
0         RANGE RESOURCES CORP  4632.14485
1                     ANGI INC  4627.03953
3                TERADATA CORP  4607.99500
6            INARI MEDICAL INC  4582.21035
9          MERITAGE HOMES CORP  4554.18066
16        JETBLUE AIRWAYS CORP  4528.76144
22                    FIGS INC  4504.90248
23            BLACK HILLS CORP  4503.77740
24  HANNON ARMSTRONG SUST INFR  4498.94528
25                LESLIE'S INC  4494.97412
['home', 'brand', 'software', 'insurance', 'equipment']


####### KMeans Cluster 1 ########
                   compustat_name      mktcap
15    SHELL MIDSTREAM PARTNERS LP  4530.70080
33               RYDER SYSTEM INC  4425.91399
158                    MATSON INC  3784.14096
176                     GATX CORP  3698.74500
255              FORWARD AIR CORP  3264.34422
272        WERNER ENTERPRISES INC  3187.69144
278         EMBARK TECHNOLOGY INC  3146.27432
296                  ARCBEST CO

['test', 'patient', 'cell', 'testing', 'clinical']


####### KMeans Cluster 4 ########
       compustat_name     mktcap
822  MALIBU BOATS INC  1432.3332
['boat', 'dealer', 'cobalt', 'outboard', 'maverick']


####### KMeans Cluster 5 ########
                   compustat_name      mktcap
36              COMMERCIAL METALS  4412.68255
163               HILLENBRAND INC  3768.44316
270                    CABOT CORP  3192.32860
331          RANPAK HOLDINGS CORP  2949.24082
366                 INGEVITY CORP  2816.80620
405                    KADANT INC  2674.48992
454     MINERALS TECHNOLOGIES INC  2435.38295
479  GCP APPLIED TECHNOLOGIES INC  2327.23162
563                   TRINSEO PLC  2037.38902
688          KRONOS WORLDWIDE INC  1734.40550
['paper', 'brand', 'packaging', 'metal', 'equipment']


####### KMeans Cluster 6 ########
             compustat_name     mktcap
134  ACUSHNET HOLDINGS CORP  3908.2804
['golf', 'titleist', 'pro', 'ball', 'club']


####### KMeans Cluster 7 ########
    

['student', 'school', 'learning', 'online', 'course']


####### KMeans Cluster 10 ########
                   compustat_name      mktcap
10                    PROGYNY INC  4553.15050
45                  LHC GROUP INC  4346.21133
127  SELECT MEDICAL HOLDINGS CORP  3943.86300
179                   CORVEL CORP  3685.76000
197         LIFESTAN HLTH GRP INC  3561.89848
201     MERIT MEDICAL SYSTEMS INC  3517.33340
207      IRHYTHM TECHNOLOGIES INC  3463.14594
233          1LIFE HEALTHCARE INC  3353.91973
251    APOLLO MEDICAL HOLDING INC  3278.89804
292              EVERCOMMERCE INC  3076.93575
['care', 'patient', 'provider', 'clinical', 'physician']


####### KMeans Cluster 11 ########
                  compustat_name      mktcap
17   SPIRIT AEROSYSTEMS HOLDINGS  4526.47523
34                  REXNORD CORP  4417.14000
35          BWX TECHNOLOGIES INC  4414.63176
47                   HEXCEL CORP  4345.70920
67        ALARM.COM HOLDINGS INC  4245.58860
70         DIVERSEY HOLDINGS LTD  4225.

In [10]:
for i, n in indGrpNames.iterrows():
    print("")
    print("")
    print("####### GICS Industry Group " + str(n.gicsIndGrp) + ": "+n.indGrp+" ########")
    print()
    ntopics = kmDf2[kmDf2.gicsIndGrp==n.gicsIndGrp].groupby(['d2vIndGrpPca']).size().reset_index()
    ntopics.columns = ['d2vIndGrpPca','n']
    ntopics['pct'] = round(100.0*ntopics['n'] / ntopics['n'].sum(),1 )
    ntopics = ntopics.merge(topicnames, on='d2vIndGrpPca')
    ntopics = ntopics.sort_values(by='n',ascending=False)
    print(ntopics[['d2vIndGrpPca','n','pct','top5words']])
    



####### GICS Industry Group 1010: Energy ########

   d2vIndGrpPca   n   pct                                          top5words
0             0  21  38.2      [home, brand, software, insurance, equipment]
5            21  19  34.5     [natural_gas, pipeline, water, gas, gathering]
1             1   9  16.4  [pipeline, refinery, transportation, crude_oil...
3            18   3   5.5                [event, gaming, coal, mine, casino]
4            20   2   3.6           [energy, power, solar, vehicle, project]
2            11   1   1.8  [equipment, component, industrial, vehicle, de...


####### GICS Industry Group 1510: Materials ########

   d2vIndGrpPca   n   pct                                          top5words
0             0  20  45.5      [home, brand, software, insurance, equipment]
1             5  12  27.3        [paper, brand, packaging, metal, equipment]
2            11   8  18.2  [equipment, component, industrial, vehicle, de...
4            18   3   6.8                [ev

   d2vIndGrpPca   n   pct                                       top5words
0             0  15  45.5   [home, brand, software, insurance, equipment]
3            14   9  27.3     [software, payment, client, security, user]
1             8   7  21.2  [content, television, programming, medium, tv]
2             9   2   6.1     [student, school, learning, online, course]


####### GICS Industry Group 5510: Utilities ########

   d2vIndGrpPca   n   pct                                          top5words
2            20  10  38.5           [energy, power, solar, vehicle, project]
0             0   9  34.6      [home, brand, software, insurance, equipment]
3            21   6  23.1     [natural_gas, pipeline, water, gas, gathering]
1            11   1   3.8  [equipment, component, industrial, vehicle, de...


####### GICS Industry Group 6010: Real Estate ########

   d2vIndGrpPca   n   pct                                       top5words
0             0  15  62.5   [home, brand, software, insur

In [11]:
for c in range(0,24):
    print("")
    print("")
    print("####### KMeans Cluster " + str(c) + ": "+str(topicnames.top5words.iloc[c])+" ########")
    print()
    ntopics = kmDf2[kmDf2.d2vIndGrpPca==c].groupby(['gicsIndGrp']).size().reset_index()
    ntopics.columns = ['gicsIndGrp','n']
    ntopics['pct'] = round(100.0*ntopics['n'] / ntopics['n'].sum(),1 )
    ntopics = ntopics.merge(indGrpNames, on='gicsIndGrp')
    ntopics = ntopics.sort_values(by='n',ascending=False)
    print(ntopics[['gicsIndGrp','n','pct','indGrp']])
    



####### KMeans Cluster 0: ['home', 'brand', 'software', 'insurance', 'equipment'] ########

    gicsIndGrp   n   pct                                          indGrp
2         2010  38  11.7                                   Capital Goods
12        3510  24   7.4                Health Care Equipment & Services
8         2550  24   7.4                                       Retailing
17        4510  22   6.8                             Software & Services
0         1010  21   6.5                                          Energy
1         1510  20   6.2                                       Materials
18        4520  16   4.9                 Technology Hardware & Equipment
6         2520  16   4.9                     Consumer Durables & Apparel
23        6010  15   4.6                                     Real Estate
3         2020  15   4.6              Commercial & Professional Services
21        5020  15   4.6                           Media & Entertainment
7         2530  12   3.7      

   gicsIndGrp   n   pct                                          indGrp
0        3510  33  89.2                Health Care Equipment & Services
1        3520   2   5.4  Pharmaceuticals, Biotechnology & Life Sciences
2        4510   1   2.7                             Software & Services
3        6010   1   2.7                                     Real Estate


####### KMeans Cluster 11: ['equipment', 'component', 'industrial', 'vehicle', 'device'] ########

    gicsIndGrp   n   pct                                    indGrp
2         2010  50  45.5                             Capital Goods
10        4520  17  15.5           Technology Hardware & Equipment
4         2510  10   9.1                  Automobiles & Components
1         1510   8   7.3                                 Materials
11        4530   8   7.3  Semiconductors & Semiconductor Equipment
12        5010   4   3.6                Telecommunication Services
3         2030   3   2.7                            Transportation
5  

In [12]:
pickle.dump( d2v, open( "d2v_model.p", "wb" ) )
pickle.dump( km_model, open( "d2v_kmeans_model.p", "wb" ) )
pickle.dump( km_model2, open( "d2v_kmeans_model2.p", "wb" ) )

print("Pickle dumped")


Pickle dumped
